# EDA (Exploratory Data Analysis)


## Load Data

In [94]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from sklearn.cluster import KMeans
import seaborn as sns
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans
server = r"SIX\SQLEXPRESS"
database = "OlistDB"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

# Load các bảng chính (điều chỉnh path theo thư mục của bạn)
dataset_dir = r'C:\Users\nghah\OneDrive\Documents\dataset'

# Đọc các file CSV bằng cách nối thư mục gốc với tên file
orders = pd.read_csv(os.path.join(dataset_dir, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(dataset_dir, 'olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(dataset_dir, 'olist_order_items_dataset.csv'))
payments = pd.read_csv(os.path.join(dataset_dir, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(dataset_dir, 'olist_order_reviews_dataset.csv'))
products = pd.read_csv(os.path.join(dataset_dir, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(dataset_dir, 'olist_sellers_dataset.csv'))
category = pd.read_csv(os.path.join(dataset_dir, 'product_category_name_translation.csv'))

# Convert timestamp columns sang datetime ngay từ đầu
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

## Overview (row, columns, dtypes)

In [3]:
tables = {
    'orders': orders, 'customers': customers, 'order_items': order_items,
    'payments': payments, 'reviews': reviews, 'products': products, 'sellers': sellers
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(df.dtypes.value_counts().to_dict())
    print("-" * 40)

orders: 99,441 rows x 8 columns
{dtype('<M8[ns]'): 5, dtype('O'): 3}
----------------------------------------
customers: 99,441 rows x 5 columns
{dtype('O'): 4, dtype('int64'): 1}
----------------------------------------
order_items: 112,650 rows x 7 columns
{dtype('O'): 4, dtype('float64'): 2, dtype('int64'): 1}
----------------------------------------
payments: 103,886 rows x 5 columns
{dtype('O'): 2, dtype('int64'): 2, dtype('float64'): 1}
----------------------------------------
reviews: 99,224 rows x 7 columns
{dtype('O'): 6, dtype('int64'): 1}
----------------------------------------
products: 32,951 rows x 9 columns
{dtype('float64'): 7, dtype('O'): 2}
----------------------------------------
sellers: 3,095 rows x 4 columns
{dtype('O'): 3, dtype('int64'): 1}
----------------------------------------


## Statistic table

In [4]:
numeric_summary = pd.DataFrame({
    'price': order_items['price'],
})['price'].describe()

# Cách nhanh cho nhiều biến cùng lúc
num_vars = {
    'price (BRL)': order_items['price'],
    'freight_value (BRL)': order_items['freight_value'],
    'payment_value (BRL)': payments['payment_value'],
    'payment_installments': payments['payment_installments'],
    'review_score (1-5)': reviews['review_score']
}

summary_rows = []
for name, series in num_vars.items():
    summary_rows.append({
        'Variable': name,
        'Mean': round(series.mean(), 2),
        'Median': round(series.median(), 2),
        'Std Dev': round(series.std(), 2),
        'Min': series.min(),
        'Max': series.max(),
        'Skewness': round(series.skew(), 2)   # <-- bổ sung so với report gốc
    })

numeric_summary_table = pd.DataFrame(summary_rows)
print(numeric_summary_table)

               Variable    Mean  Median  Std Dev   Min       Max  Skewness
0           price (BRL)  120.65   74.99   183.63  0.85   6735.00      7.92
1   freight_value (BRL)   19.99   16.26    15.81  0.00    409.68      5.64
2   payment_value (BRL)  154.10  100.00   217.49  0.00  13664.08      9.25
3  payment_installments    2.85    1.00     2.69  0.00     24.00      1.66
4    review_score (1-5)    4.09    5.00     1.35  1.00      5.00     -1.36


## statistic classification table

In [5]:
cat_vars = {
    'order_status': orders['order_status'],
    'payment_type': payments['payment_type'],
    'customer_state': customers['customer_state'],
    'seller_state': sellers['seller_state'],
    'review_score': reviews['review_score']
}

cat_summary_rows = []
for name, series in cat_vars.items():
    vc = series.value_counts()
    mode_val = vc.index[0]
    freq = vc.iloc[0]
    pct = round(freq / len(series) * 100, 1)
    cat_summary_rows.append({
        'Variable': name,
        'Unique Count': series.nunique(),
        'Mode': mode_val,
        'Frequency': f"{freq:,} ({pct}%)"
    })

cat_summary_table = pd.DataFrame(cat_summary_rows)
print(cat_summary_table)

         Variable  Unique Count         Mode       Frequency
0    order_status             8    delivered  96,478 (97.0%)
1    payment_type             5  credit_card  76,795 (73.9%)
2  customer_state            27           SP  41,746 (42.0%)
3    seller_state            23           SP   1,849 (59.7%)
4    review_score             5            5  57,328 (57.8%)


## Check duplication

In [6]:
print("Duplicate order_id:", orders['order_id'].duplicated().sum())
print("Duplicate customer_unique_id rows:", customers.duplicated().sum())
print("Fully duplicated rows in order_items:", order_items.duplicated().sum())

Duplicate order_id: 0
Duplicate customer_unique_id rows: 0
Fully duplicated rows in order_items: 0


## Data Volume follow by year

In [7]:
orders['order_year'] = orders['order_purchase_timestamp'].dt.year
yearly_volume = orders['order_year'].value_counts().sort_index()
yearly_pct = (yearly_volume / yearly_volume.sum() * 100).round(1)

volume_by_year = pd.DataFrame({
    'Order Count': yearly_volume,
    'Percentage': yearly_pct
})
print(volume_by_year)

            Order Count  Percentage
order_year                         
2016                329         0.3
2017              45101        45.4
2018              54011        54.3


## Check outlier in EDA

In [8]:
# Price outliers theo IQR
Q1, Q3 = order_items['price'].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print(f"Price upper bound (IQR): {upper_bound:.2f} BRL")
print(f"Số orders vượt ngưỡng: {(order_items['price'] > upper_bound).sum()}")
print(f"Max price: {order_items['price'].max()}")  # để đối chiếu với 6,735 BRL trong báo cáo

# Payment installments = 0 (bất thường nghiệp vụ)
zero_installments = (payments['payment_installments'] == 0).sum()
print(f"Payments với installments = 0: {zero_installments}")

Price upper bound (IQR): 277.40 BRL
Số orders vượt ngưỡng: 8427
Max price: 6735.0
Payments với installments = 0: 2


## Missing Value Overview

In [9]:
def missing_report(df, table_name):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({'NULL Count': missing, 'NULL %': missing_pct})
    report = report[report['NULL Count'] > 0]
    report['Table'] = table_name
    return report

all_missing = pd.concat([
    missing_report(orders, 'orders'),
    missing_report(products, 'products'),
    missing_report(reviews, 'reviews')
])
print(all_missing)

                               NULL Count  NULL %     Table
order_approved_at                     160    0.16    orders
order_delivered_carrier_date         1783    1.79    orders
order_delivered_customer_date        2965    2.98    orders
product_category_name                 610    1.85  products
product_name_lenght                   610    1.85  products
product_description_lenght            610    1.85  products
product_photos_qty                    610    1.85  products
product_weight_g                        2    0.01  products
product_length_cm                       2    0.01  products
product_height_cm                       2    0.01  products
product_width_cm                        2    0.01  products
review_comment_title                87656   88.34   reviews
review_comment_message              58247   58.70   reviews


### Visisor quantity by code

In [10]:
# Review score bimodal check
review_dist = reviews['review_score'].value_counts(normalize=True).sort_index() * 100
print(review_dist.round(1))

# Late delivery rate
delivered = orders.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date'])
late_rate = (delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']).mean() * 100
print(f"Late delivery rate: {late_rate:.1f}%")

review_score
1    11.5
2     3.2
3     8.2
4    19.3
5    57.8
Name: proportion, dtype: float64
Late delivery rate: 8.1%
